# Simplest Test

## Pyomo Model and function Definitions


In [1]:
import numpy as np
from scipy import sparse 
import sys
from pyomo.environ import * 

In [2]:
def pyomo_optimize_MILP(matricies, f, prev_Candidates: dict = {}, pSlack = False):
    # note: what I am doing here is rebuilding the Model each time I call this function. And each time I call the function I have a different set of Prev Candidate. 
    # What makes far more sense is that I have the Candidate saving occuring after each solve and just adding 1 more set of constraints as opposed to rebuilding the model each time.     
    
    A,B,C = matricies

    
    def sparse_mat_to_dict(A):
        matrix_dict = {(i-1, A.indices[j]): A.data[j] for i in range(1,A.shape[0]+1) for j in range (A.indptr[i-1],A.indptr[i])}
        return matrix_dict
    
    def sparse_vec_to_dict(v):
        vec_dict = { i-1:v.data[j] for i in range(1,v.shape[0]+1) for j in range(v.indptr[i-1], v.indptr[i])}
        return vec_dict
    
    techno_dict = sparse_mat_to_dict(A)
    bio_dict = sparse_mat_to_dict(B)
    method_dict = sparse_mat_to_dict(C)

    Product = {None: list({i[0] for i in techno_dict})}
    Process = {None: list({i[1] for i in techno_dict})}
    Product_Process = {None: list({(i[0],i[1]) for i in techno_dict})}
    Bio_flows = {None: list({i[0] for i in bio_dict})}
    Bio_process = {None: list({i for i in bio_dict})}
    Method = {None: list({i for i in method_dict})}

    # create model
    M = ConcreteModel()

    # create sets
    M.Product = Set(initialize=Product)
    M.Process = Set(initialize=Process)
    M.Product_process = Set(within=M.Product * M.Process, initialize=Product_Process)
    M.Bio_flows = Set(initialize=Bio_flows)
    M.Bio_process = Set(within=M.Bio_flows * M.Process, initialize=Bio_process)
    M.Method = Set(within=M.Bio_flows * M.Bio_flows, initialize=Method)
    
    M.Bio_in = Set(within=Bio_flows, doc='This Bioflow contributes to this Impact(like GWP)')
    M.Bio_out = Set(M.Bio_flows, within = M.Process, doc='This process generates this Bioflow which in used to calculate this Impact')
    M.Process_out = Set(M.Product, within= M.Process, doc='this process produces or uses this product')
    M.Process_in = Set(M.Process, within= M.Product, doc='this product is used in this process')

    def populate_env(M):
        for i,j in M.Bio_process:
            if i not in M.Bio_in:
                M.Bio_in.add(i)
            M.Bio_out[i].add(j)

    def populate_in_out(M):
        for i,j in M.Product_process:
            M.Process_in[j].add(i)
            M.Process_out[i].add(j)


    M.Env = BuildAction(rule=populate_env)
    M.In_out = BuildAction(rule= populate_in_out)

    # create variable
    # Add bounds and maybe 
    # M.scaling = Var(M.Process, bounds=(0,1e24))
    M.scaling = Var(M.Process, bounds=(0,1e24))
    
    M.scaling_lb = Param(default=1e-3)
    M.scaling_ub = Param(default= 1e10)

    M.f_ = Var(M.Product, bounds=(-1e10, 1e10))
    M.y_s = Var(M.Process, domain=Binary)
    M.pSlack = Var(M.Product, bounds=(0, 1e10))

    M.A = Param(M.Product_process, initialize=techno_dict, default=0, mutable=True)
    M.B = Param(M.Bio_process, initialize=bio_dict, default=0, mutable=True)
    M.C = Param(M.Method, initialize=method_dict, default=0, mutable=True)
    M.f = Param(M.Product, initialize=sparse_vec_to_dict(f), default=0, mutable=True)
    

    def bin_lb(M,j):
        return  M.y_s[j] * M.scaling_lb <= M.scaling[j]
    M.bin_lb_con = Constraint(M.Process, rule = bin_lb)

    def bin_ub(M,j): 
        return M.scaling[j] <= M.y_s[j] * M.scaling_ub
    M.bin_ub_con = Constraint(M.Process, rule = bin_ub)

    
    def equality_rule(M, i):
        return sum( M.A[i,j] * M.scaling[j] for j in M.Process_out[i]  ) == M.f[i]
    
    def equality_rule_pos(M, i):
        return sum( M.A[i,j] * M.scaling[j] for j in M.Process_out[i]  ) == M.f[i] +  M.pSlack[i]

# todo: see if I want to use the 
    def obj(M):
        return sum(M.C[e,e] * sum(M.B[e,p] * M.scaling[p] for p in M.Bio_out[e]) for e in M.Bio_in) + sum(M.y_s[j] for j in M.Process)
    
    def obj_pSlack(M):
        return sum(M.C[e,e] * sum(M.B[e,p] * M.scaling[p] for p in M.Bio_out[e]) for e in M.Bio_in) + sum(M.y_s[j] for j in M.Process) + 9999*sum(M.pSlack[i] for i in M.Product)

        
    # Changes from base Model
    if pSlack:
        M.eq_con = Constraint(M.Product, rule=equality_rule_pos)
        M.obj = Objective(rule=obj_pSlack)
    else:
        M.eq_con = Constraint(M.Product, rule=equality_rule)
        M.obj = Objective(rule=obj)

# Here for the integer cuts   
    if len(prev_Candidates) != 0:
        cut_dict = prev_Candidates["A_cut"]
        N_0_dict = prev_Candidates["N_0"]

        #Todo: I am certain that this is really slow..... ask lucas maybe?
        M.Cut_set = RangeSet(0,len(list(set(i[0] for i in cut_dict)))-1)
                             
        M.Cuts = Param(M.Cut_set, M.Process, initialize=cut_dict, default=-1, mutable = True)
        M.N_0 = Param(M.Cut_set, initialize=N_0_dict, mutable=True)

        def integer_cut(M,c):
            return sum( M.Cuts[c,j] * M.y_s[j] for j in M.Process) <= len(Process[None]) - 1 - M.N_0[c] 
        M.int_cuts = Constraint(M.Cut_set, rule=integer_cut)

    pyomo.common.Executable('gams').set_path("C:/GAMS/37/gams.exe")
    opt=SolverFactory('gams')
    # opt = SolverFactory('gurobi')

    results = opt.solve(M, solver='cplex', tee=True)

    return M

In [3]:
def get_N_candidates(Candidiates):
    return len(list(set(i[0] for i in Candidiates["A_cut"])))

def add_opt_candidate(Model, Candidiates=None):
        
    if Candidiates == None:
        Candidiates = {"A_cut":{}, "N_0":{}}

    N = len(Model.scaling)
        
    index = get_N_candidates(Candidiates)
    for i in range(len(Model.scaling)):
        if round(Model.y_s[i].value,2) == 1.0:
            Candidiates['A_cut'][index,i] = 1
            N -= 1

    Candidiates['N_0'][index] = N
    
    return Candidiates

## Example Calculation


In [4]:
A_np = np.array([1, 1, 1])
A = sparse.csr_matrix(A_np)
A.toarray()

array([[1, 1, 1]], dtype=int32)

In [5]:
B_np = np.array([1, 2, 3])
B = sparse.csr_matrix(B_np)
B.toarray()

array([[1, 2, 3]], dtype=int32)

In [6]:
C_np = np.array([1])
C = sparse.csr_matrix(C_np)
C.toarray()

array([[1]], dtype=int32)

In [7]:
f_np = np.zeros([A.shape[0], 1])
f_np[-1] = 1
f = sparse.csr_matrix(f_np)
f.toarray()

array([[1.]])

In [8]:
Model = pyomo_optimize_MILP((A,B,C),f)
print("\n---------------------------")
print("scaling, y")
for i in range(len(Model.scaling)):
    print(Model.scaling[i].value, Model.y_s[i].value)

--- Job model.gms Start 10/23/23 09:58:06 37.1.0 r07954d5 WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\GAMS\37\gmsprmNT.txt
    C:\Users\Usuario\Documents\GAMS\gamsconfig.yaml
--- GAMS Parameters defined
    MIP CPLEX
    Input C:\Users\Usuario\AppData\Local\Temp\tmpousjxosi\model.gms
    Output C:\Users\Usuario\AppData\Local\Temp\tmpousjxosi\output.lst
    ScrDir C:\Users\Usuario\AppData\Local\Temp\tmpousjxosi\225a\
    SysDir C:\GAMS\37\
    CurDir C:\Users\Usuario\AppData\Local\Temp\tmpousjxosi\
    LogOption 3
    OptCR 1E-9
Licensee: Antonio Espuna, Single User License            S210319|0002AN-WIN
          Universitat Politecnica de Catalunya, Chemical Engineering DC6757
          C:\Users\Usuario\Documents\GAMS\gamslice.txt
          antonio.espuna@upc.edu                                           
Processor information: 1 socket(s), 12 core(s), and 20 thread(s) available
GAMS 37.1.0   Copyright (C) 1987-2021 GAMS Development. All rights reserved
--- Starting compilation
-

In [9]:
Candidiates= {"A_cut":{}, 'N_0':{}}

Candidiates = add_opt_candidate(Model, Candidiates)
print(Candidiates)

{'A_cut': {(0, 0): 1}, 'N_0': {0: 2}}


In [10]:
Model = pyomo_optimize_MILP((A,B,C),f,Candidiates)
print("\n---------------------------")
print("scaling, y")
for i in range(len(Model.scaling)):
    print(Model.scaling[i].value, Model.y_s[i].value)

--- Job model.gms Start 10/23/23 09:58:06 37.1.0 r07954d5 WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\GAMS\37\gmsprmNT.txt
    C:\Users\Usuario\Documents\GAMS\gamsconfig.yaml
--- GAMS Parameters defined
    MIP CPLEX
    Input C:\Users\Usuario\AppData\Local\Temp\tmpwapdigm7\model.gms
    Output C:\Users\Usuario\AppData\Local\Temp\tmpwapdigm7\output.lst
    ScrDir C:\Users\Usuario\AppData\Local\Temp\tmpwapdigm7\225a\
    SysDir C:\GAMS\37\
    CurDir C:\Users\Usuario\AppData\Local\Temp\tmpwapdigm7\
    LogOption 3
    OptCR 1E-9
Licensee: Antonio Espuna, Single User License            S210319|0002AN-WIN
          Universitat Politecnica de Catalunya, Chemical Engineering DC6757
          C:\Users\Usuario\Documents\GAMS\gamslice.txt
          antonio.espuna@upc.edu                                           
Processor information: 1 socket(s), 12 core(s), and 20 thread(s) available
GAMS 37.1.0   Copyright (C) 1987-2021 GAMS Development. All rights reserved
--- Starting compilation
-

In [11]:
Candidiates = add_opt_candidate(Model, Candidiates)
Candidiates

{'A_cut': {(0, 0): 1, (1, 1): 1}, 'N_0': {0: 2, 1: 2}}

In [12]:
Model = pyomo_optimize_MILP((A,B,C),f,Candidiates)
print("\n---------------------------")
print("scaling, y")
for i in range(len(Model.scaling)):
    print(Model.scaling[i].value, Model.y_s[i].value)

--- Job model.gms Start 10/23/23 09:58:06 37.1.0 r07954d5 WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\GAMS\37\gmsprmNT.txt
    C:\Users\Usuario\Documents\GAMS\gamsconfig.yaml
--- GAMS Parameters defined
    MIP CPLEX
    Input C:\Users\Usuario\AppData\Local\Temp\tmp2_svs04a\model.gms
    Output C:\Users\Usuario\AppData\Local\Temp\tmp2_svs04a\output.lst
    ScrDir C:\Users\Usuario\AppData\Local\Temp\tmp2_svs04a\225a\
    SysDir C:\GAMS\37\
    CurDir C:\Users\Usuario\AppData\Local\Temp\tmp2_svs04a\
    LogOption 3
    OptCR 1E-9
Licensee: Antonio Espuna, Single User License            S210319|0002AN-WIN
          Universitat Politecnica de Catalunya, Chemical Engineering DC6757
          C:\Users\Usuario\Documents\GAMS\gamslice.txt
          antonio.espuna@upc.edu                                           
Processor information: 1 socket(s), 12 core(s), and 20 thread(s) available
GAMS 37.1.0   Copyright (C) 1987-2021 GAMS Development. All rights reserved
--- Starting compilation
-